# Merge & Report

Run this notebook **after** notebooks 1 till 5 have all finished.

It:
1. Copies the five method cache files from Drive into the local cache
2. Runs the full experiment (all methods) — every result is a cache hit, no timing
3. Generates the interactive HTML report
4. Saves the HTML to Drive and offers a download link

### Expected Drive folder contents before running
```
MyDrive/treebranchmarks/woodelfhd_sweep/results_jsons/
  woodelf_hd.json              ← written by notebook 01
  original_woodelf.json        ← written by notebook 02
  shap.json                    ← written by notebook 03
  woodelf_hd_gpu.json          ← written by notebook 04
  pltreeshap_fasttreeshap.json ← written by notebook 05
```

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/')
REPORT_HTML  = DRIVE_FOLDER / 'woodelfhd_main_experiment.html'

DRIVE_CACHES = {
    'woodelf_hd':              DRIVE_FOLDER / 'results_jsons' / 'woodelf_hd.json',
    'original_woodelf':        DRIVE_FOLDER / 'results_jsons' / 'original_woodelf.json',
    'shap':                    DRIVE_FOLDER / 'results_jsons' / 'shap.json',
    'pltreeshap_fasttreeshap': DRIVE_FOLDER / 'results_jsons' / 'pltreeshap_fasttreeshap.json',
    'woodelf_hd_gpu':          DRIVE_FOLDER / 'results_jsons' / 'woodelf_hd_gpu.json',
}

missing = [str(p) for p in DRIVE_CACHES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'The following method cache files are missing — run the corresponding '
        'notebooks first:\n' + '\n'.join(missing)
    )
print('All five method cache files found.')

All five method cache files found.


In [ ]:
# ── Step 3: Clone treebranchmarks and install ────────────────────────────────
# woodelf_explainer is needed by treebranchmarks even though this notebook
# only calls HtmlGenerator (which does not import woodelf directly).

TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

!pip install woodelf_explainer
!pip install -q -e /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 543, done.
remote: Counting objects: 100% (543/543), done.
remote: Compressing objects: 100% (334/334), done.
remote: Total 543 (delta 355), reused 378 (delta 193), pack-reused 0 (from 0)
Receiving objects: 100% (543/543), 372.04 KiB | 20.67 MiB/s, done.
Resolving deltas: 100% (355/355), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.5 MB/s eta 0:00:00
  Building editable for treebranchmarks (pyproject.toml) ... done


In [ ]:
# ── Step 4: Copy method cache files into local cache ─────────────────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_main_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)

for method_name, drive_path in DRIVE_CACHES.items():
    dest = cache_dir / f'{method_name}.json'
    shutil.copy(drive_path, dest)
    print(f'Copied {method_name}.json ({drive_path.stat().st_size // 1024} KB)')

Copied woodelf_hd.json (43 KB)
Copied original_woodelf.json (23 KB)
Copied shap.json (62 KB)
Copied pltreeshap_fasttreeshap.json (27 KB)
Copied woodelf_hd_gpu.json (60 KB)


In [ ]:
# ── Step 5: Run experiment (all cache hits) + generate HTML ──────────────────
import os, shutil

os.chdir('/content/treebranchmarks')

from benchmarks.woodelfhd_main_experiment import build_experiment

exp = build_experiment()
exp.run()
html_path = exp.generate_html()

shutil.copy(html_path, REPORT_HTML)
print(f'HTML report saved to Drive: {REPORT_HTML}')


Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']
[dataset:fraud_detection] Cache miss — downloading and preprocessing.


Downloading...
From: https://drive.google.com/uc?id=1A1Qdtron9XtZ6h85uNdaFCprUkH5KA5P
To: /content/treebranchmarks/cache/datasets/fraud_detection/raw/data.parquet
100%|██████████| 69.6M/69.6M [00:00<00:00, 130MB/s]


[dataset:fraud_detection] Cached 590540 rows × 397 features.

  > D=6  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 10.58s — T=100, D=6, L=32.0, F=397
  [approach:WoodelfHD] CACHED=3.409s
  [approach:OriginalWoodelf] CACHED=6.611s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=43.158s
  [approach:SHAP] CACHED=14.012s

  > D=9  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 13.22s — T=100, D=9, L=82.8, F=397
  [approach:WoodelfHD] CACHED=12.948s
  [approach:OriginalWoodelf] CACHED=69.878s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=156.261s
  [approach:SHAP] CACHED=66.733s

  > D=12  n=118108  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 16.02s — T=100, D=12, L=154.0, F=397
  [approach:WoodelfHD] CACHED=35.725s
[model:lightgbm] Training.
[model:lightgbm] Trained in 6.81s — T=10, D=12, L=202.5, F=397
  [approach:OriginalWoodelf] CACHED=4282.393s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=511.303s
  [approach:SHAP] CACHED=186

Downloading...
From (original): https://drive.google.com/uc?id=1iDX-Uxs4SruwpjoKjDrn9DW80wkFid83
From (redirected): https://drive.google.com/uc?id=1iDX-Uxs4SruwpjoKjDrn9DW80wkFid83&confirm=t&uuid=18f89bb4-530e-41c2-9f36-03f9ece277b5
To: /content/treebranchmarks/cache/datasets/higgs/raw/higgs.parquet
100%|██████████| 898M/898M [00:06<00:00, 148MB/s]


[dataset:higgs] Cached 11000000 rows × 28 features.

  > D=6  n=2200000  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 57.36s — T=100, D=6, L=62.8, F=28
  [approach:WoodelfHD] CACHED=194.520s
  [approach:OriginalWoodelf] CACHED=82.518s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=1055.480s
  [approach:SHAP] CACHED=390.294s

  > D=9  n=2200000  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 17.33s — T=10, D=9, L=485.6, F=28
  [approach:WoodelfHD] CACHED=2202.127s
  [approach:OriginalWoodelf] CACHED=2185.502s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=14764.124s
  [approach:SHAP] CACHED=4879.299s

  > D=12  n=2200000  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 9.36s — T=1, D=12, L=2024.0, F=28
  [approach:WoodelfHD] CACHED=11768.521s
  [approach:OriginalWoodelf] MEMORY CRASH (configured)
  [approach:PLTreeSHAP + FastTreeSHAP] MEMORY CRASH (configured)
  [approach:SHAP] CACHED=28165.702s

  > D=15  n=2200000  m=0
[model:lightgbm] Training.
[mod

Downloading...
From: https://drive.google.com/uc?id=1sExGIsElOZFEJUv5f42EJzvQZZk8shUg
To: /content/treebranchmarks/cache/datasets/intrusion_detection/raw/data.parquet
100%|██████████| 10.5M/10.5M [00:00<00:00, 82.2MB/s]


[dataset:intrusion_detection] Cached 4898431 rows × 121 features.

  > D=6  n=2984154  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 14.52s — T=100, D=6, L=33.8, F=121
  [approach:WoodelfHD] CACHED=166.818s
  [approach:OriginalWoodelf] CACHED=74.312s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=434.675s
  [approach:SHAP] CACHED=286.812s

  > D=9  n=2984154  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 4.68s — T=10, D=9, L=62.3, F=121
  [approach:WoodelfHD] CACHED=439.572s
  [approach:OriginalWoodelf] CACHED=259.519s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=880.600s
  [approach:SHAP] CACHED=1052.166s

  > D=12  n=2984154  m=0
[model:lightgbm] Training.
[model:lightgbm] Trained in 5.28s — T=10, D=12, L=117.2, F=121
  [approach:WoodelfHD] CACHED=1053.571s
  [approach:OriginalWoodelf] CACHED=1040.152s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=1117.317s
  [approach:SHAP] CACHED=2831.430s

  > D=15  n=2984154  m=0
[model:lightgbm] Training.
[model:lightgb

In [ ]:
# ── Step 4: Copy method cache files into local cache ─────────────────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)

for method_name, drive_path in DRIVE_CACHES.items():
    dest = cache_dir / f'{method_name}.json'
    shutil.copy(drive_path, dest)
    print(f'Copied {method_name}.json ({drive_path.stat().st_size // 1024} KB)')

Copied woodelf_hd.json (43 KB)
Copied original_woodelf.json (23 KB)
Copied shap.json (62 KB)
Copied pltreeshap_fasttreeshap.json (27 KB)
Copied woodelf_hd_gpu.json (60 KB)


In [ ]:
# ── Step 5: Run experiment (all cache hits) + generate HTML ──────────────────
import os, shutil

os.chdir('/content/treebranchmarks')

from benchmarks.woodelfhd_depth_sweep_experiment import build_experiment

exp = build_experiment()
exp.run()
html_path = exp.generate_html()

REPORT_WITH_WOODELF_GPU_HTML = DRIVE_FOLDER / 'woodelfhd_depth_sweep_experiment.html'
shutil.copy(html_path, REPORT_WITH_WOODELF_GPU_HTML)
print(f'HTML report saved to Drive: {REPORT_WITH_WOODELF_GPU_HTML}')


Mission: fraud_detection PD SHAP sweep_D
  dataset   : fraud_detection
  D_values  : [6, 9, 12, 15, 18, 21]
  n=118108  m=0
  tasks     : ['Path-Dependent SHAP']

  > D=6  n=118108  m=0
  [approach:WoodelfHD] CACHED=3.409s
  [approach:WoodelfHD GPU] CACHED=15.049s
  [approach:OriginalWoodelf] CACHED=6.611s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=43.158s
  [approach:SHAP] CACHED=14.012s

  > D=9  n=118108  m=0
  [approach:WoodelfHD] CACHED=12.948s
  [approach:WoodelfHD GPU] CACHED=41.610s
  [approach:OriginalWoodelf] CACHED=69.878s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=156.261s
  [approach:SHAP] CACHED=66.733s

  > D=12  n=118108  m=0
  [approach:WoodelfHD] CACHED=35.725s
  [approach:WoodelfHD GPU] CACHED=98.236s
  [approach:OriginalWoodelf] CACHED=4282.393s
  [approach:PLTreeSHAP + FastTreeSHAP] CACHED=511.303s
  [approach:SHAP] CACHED=186.604s

  > D=15  n=118108  m=0
  [approach:WoodelfHD] CACHED=146.636s
  [approach:WoodelfHD GPU] CACHED=182.714s
  [approach:OriginalW

In [ ]:
# ── Step 6: Download the HTML report ─────────────────────────────────────────
from google.colab import files
files.download(str(REPORT_HTML))
files.download(str(REPORT_WITH_WOODELF_GPU_HTML))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>